In [ ]:
!nvidia-smi

Sun Apr 26 15:25:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
%%writefile device_query.cu
#include <stdio.h>
#include <cuda_runtime.h>

int main() {
    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);

    printf("GPU: %s\n", prop.name);
    printf("Compute Capability: %d.%d\n", prop.major, prop.minor);

    printf("Clock Rate: %.2f GHz\n", prop.clockRate / 1e6);
    printf("Memory Clock Rate: %.2f GHz\n", prop.memoryClockRate / 1e6);
    printf("Memory Bus Width: %d bits\n", prop.memoryBusWidth);

    // Bandwidth calculation
    float bandwidth = 2.0 * prop.memoryClockRate * (prop.memoryBusWidth / 8) / 1e6;
    printf("Theoretical Memory Bandwidth: %.2f GB/s\n", bandwidth);

    printf("Max Threads per Block: %d\n", prop.maxThreadsPerBlock);
    printf("Warp Size: %d\n", prop.warpSize);

    printf("Shared Memory per Block: %lu KB\n", prop.sharedMemPerBlock / 1024);
    printf("Global Memory: %.2f GB\n", prop.totalGlobalMem / 1e9);

    return 0;
}

Overwriting device_query.cu


In [ ]:
!nvcc device_query.cu -o device_query
!./device_query

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
GPU: Tesla T4
Compute Capability: 7.5
Clock Rate: 1.59 GHz
Memory Clock Rate: 5.00 GHz
Memory Bus Width: 256 bits
Theoretical Memory Bandwidth: 320.06 GB/s
Max Threads per Block: 1024
Warp Size: 32
Shared Memory per Block: 48 KB
Global Memory: 15.64 GB


In [ ]:
%%writefile array_sum.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void sumKernel(float *input, float *result, int n) {
    __shared__ float cache[256];

    int tid = threadIdx.x + blockIdx.x * blockDim.x;
    int cacheIndex = threadIdx.x;

    float temp = 0;
    while (tid < n) {
        temp += input[tid];
        tid += blockDim.x * gridDim.x;
    }

    cache[cacheIndex] = temp;
    __syncthreads();

    for (int i = blockDim.x/2; i != 0; i /= 2) {
        if (cacheIndex < i)
            cache[cacheIndex] += cache[cacheIndex + i];
        __syncthreads();
    }

    if (cacheIndex == 0)
        atomicAdd(result, cache[0]);
}

int main() {
    int n = 1<<20; // 1M elements
    size_t size = n * sizeof(float);

    float *h_input = (float*)malloc(size);
    for (int i = 0; i < n; i++) h_input[i] = 1.0f;

    float *d_input, *d_result;
    float result = 0;

    cudaMalloc(&d_input, size);
    cudaMalloc(&d_result, sizeof(float));

    cudaMemcpy(d_input, h_input, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &result, sizeof(float), cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<256,256>>>(d_input, d_result, n);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    cudaMemcpy(&result, d_result, sizeof(float), cudaMemcpyDeviceToHost);

    printf("Sum = %f\n", result);
    printf("Time = %f ms\n", milliseconds);

    // Bandwidth calculation
    float bandwidth = (size / (milliseconds / 1000.0)) / 1e9;
    printf("Effective Bandwidth = %f GB/s\n", bandwidth);

    cudaFree(d_input);
    cudaFree(d_result);
    free(h_input);
}

Overwriting array_sum.cu


In [ ]:
!nvcc array_sum.cu -o array_sum
!./array_sum

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Sum = 1048576.000000
Time = 16.715488 ms
Effective Bandwidth = 0.250923 GB/s


In [ ]:
%%writefile matrix_add.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void matrixAdd(int *A, int *B, int *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        int idx = row * N + col;
        C[idx] = A[idx] + B[idx];
    }
}

int main() {
    int N = 1024;
    size_t size = N * N * sizeof(int);

    int *h_A = (int*)malloc(size);
    int *h_B = (int*)malloc(size);
    int *h_C = (int*)malloc(size);

    for (int i = 0; i < N*N; i++) {
        h_A[i] = 1;
        h_B[i] = 2;
    }

    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 threads(16,16);
    dim3 blocks(N/16, N/16);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    matrixAdd<<<blocks, threads>>>(d_A, d_B, d_C, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms;
    cudaEventElapsedTime(&ms, start, stop);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    printf("Time = %f ms\n", ms);

    // FLOPS
    float flops = (float)(N*N) / (ms / 1000.0) / 1e9;
    printf("GFLOPS = %f\n", flops);

    // Memory bandwidth
    float bytes = 3 * size; // 2 reads + 1 write
    float bandwidth = bytes / (ms / 1000.0) / 1e9;
    printf("Bandwidth = %f GB/s\n", bandwidth);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

Writing matrix_add.cu


In [ ]:
!nvcc matrix_add.cu -o matrix_add
!./matrix_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Time = 0.194464 ms
GFLOPS = 5.392134
Bandwidth = 64.705612 GB/s
